# Geospatial Visualization with Python

This notebook shows four visualization patterns: **raster data**, **vector data**, **combining raster and vector** layers, and **3D objects**. We use [leafmap](https://leafmap.org/) for interactive 2D and 3D maps.

In [6]:
# Optional: Install packages if needed
# %pip install leafmap geopandas rasterio

import leafmap
import geopandas as gpd
from pathlib import Path
import numpy as np

## Example 1: Visualizing Raster Data

Raster visualization displays pixel values as colors. For single-band data (e.g., elevation), we use a colormap. For multi-band RGB imagery, we display bands 1–3 as red, green, and blue.

In [17]:
# Use a remote Cloud Optimized GeoTIFF (COG) for raster display
cog_url = "https://github.com/opengeos/data/releases/download/raster/Libya-2023-07-01.tif"

In [18]:
# Visualize the raster with leafmap
m = leafmap.Map()
m.add_cog_layer(cog_url, name="Satellite imagery (Libya)", bands=["b1", "b2", "b3"])
m

Map(center=[22.628278391361995, 32.774842775326846], controls=(ZoomControl(options=['position', 'zoom_in_text'…

**Interpretation**: `add_cog_layer` streams the raster from a URL without downloading the full file. Bands `b1`, `b2`, `b3` are displayed as RGB. The map auto-centers on the COG extent.

## Example 2: Visualizing Vector Data

Vector visualization draws points, lines, and polygons on a map. We can color features by an attribute (e.g., population, land use) or use a single style for all features.

In [19]:
from shapely.geometry import Point, Polygon

# Create sample vector data in WGS84 (lat/lon) for leafmap display
points = [
    Point(-122.42, 37.78), Point(-122.40, 37.79), Point(-122.38, 37.78),
    Point(-122.40, 37.80), Point(-122.36, 37.79)
]
polygons = [
    Polygon([(-122.44, 37.76), (-122.42, 37.76), (-122.42, 37.78), (-122.44, 37.78), (-122.44, 37.76)]),
    Polygon([(-122.40, 37.78), (-122.38, 37.78), (-122.38, 37.80), (-122.40, 37.80), (-122.40, 37.78)]),
]
gdf_points = gpd.GeoDataFrame(
    {"id": range(len(points)), "value": [10, 20, 15, 25, 30]},
    geometry=points, crs="EPSG:4326"
)
gdf_polygons = gpd.GeoDataFrame(
    {"id": range(len(polygons)), "type": ["A", "B"]},
    geometry=polygons, crs="EPSG:4326"
)

In [20]:
# Visualize vector data with leafmap
m = leafmap.Map(center=[37.78, -122.40], zoom=12)
m.add_geojson(gdf_polygons.to_json(), layer_name="Polygons", style={"fillColor": "lightblue", "color": "darkblue", "fillOpacity": 0.6})
m.add_geojson(gdf_points.to_json(), layer_name="Points", style={"color": "red", "weight": 3})
m

Map(center=[37.78, -122.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

**Interpretation**: Leafmap displays polygons and points as interactive layers. Use the layer control to toggle visibility. This pattern is useful for choropleth maps and thematic overlays.

## Example 3: Combining Raster and Vector Data

Overlaying vector features on a raster base map is a common workflow: e.g., building footprints on satellite imagery, or administrative boundaries on a land cover map. Both layers must share the same CRS.

In [21]:
# Use the same vector data (already in WGS84)
gdf_polygons_match = gdf_polygons
gdf_points_match = gdf_points

In [26]:
# Combine raster and vector with leafmap
m = leafmap.Map(center=[37.78, -122.40], zoom=12)
m.add_basemap("Esri.WorldImagery")  # Satellite basemap as raster backdrop
m.add_geojson(gdf_polygons_match.to_json(), layer_name="Polygons", style={"color": "red", "weight": 2, "fillOpacity": 0})
m.add_geojson(gdf_points_match.to_json(), layer_name="Points", style={"color": "yellow", "weight": 2})
m

Map(center=[37.78, -122.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

**Interpretation**: The satellite basemap provides the raster backdrop; vector layers are drawn on top. Red polygon outlines and yellow points show how features align with the imagery. In real applications, you might overlay building footprints on aerial imagery or roads on a land cover map.

## Example 4: Displaying 3D Objects

Leafmap's **pydeck** backend supports 3D visualization. Polygons can be **extruded** (given height) based on an attribute, and points can be shown as 3D columns. Use `import leafmap.deck as leafmap` and set `extruded=True` with `get_elevation` to map an attribute to height. Press **Ctrl + left mouse** to rotate the 3D view.

**Interpretation**: Each state polygon is extruded vertically by its land area (`ALAND`). The 3D view reveals spatial patterns that are harder to see in 2D. Use `elevation_scale` to adjust the height exaggeration.

In [7]:
# 3D column layer: points with height (funding amounts as column height)
import geopandas as gpd
import leafmap.deck as leafmap_deck
m = leafmap_deck.Map(center=(40, -100), zoom=3)
gdf_3d = gpd.read_file(
    "https://data.source.coop/cboettig/conservation-policy/Inflation_Reduction_Act_Projects.geojson"
)
# Keep only rows with valid funding for 3D extrusion
gdf_3d = gdf_3d[gdf_3d["FUNDING_NUMERIC"].notna()]
m.add_vector(
    gdf_3d,
    layer_type="ColumnLayer",
    get_position=["LONGITUDE", "LATITUDE"],
    get_elevation="FUNDING_NUMERIC",
    get_fill_color=[255, 200, 0, 180],
    elevation_scale=0.01,
    radius=15000,
    pickable=True,
)
m

{
  "initialViewState": {
    "latitude": 40,
    "longitude": -100,
    "zoom": 3
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "data": [
        {
          "ADDRESS_GEOCODE": "Madison, WI",
          "AGENCY_NAME": "Environmental Protection Agency",
          "BUREAU_NAME": "Office of Air and Radiation",
          "CATEGORY": "Environmental Remediation",
          "CITY": "Madison",
          "COUNTY": null,
          "FUNDING": "429746",
          "FUNDING_NUMERIC": 429746.0,
          "FUNDING_SOURCE": "IRA",
          "ID": "S-128",
          "LATITUDE": 43.07313,
          "LONGITUDE": -89.38644,
          "OBJECTID": 1,
          "PROGRAM_NAME": "Funding to Address Air Pollution: Fenceline Air Monitoring",
          "PROJECT_NAME": "City of Madison",
          "STATE_ABBR": "WI",
          "STATE_NAME": "Wisconsin",
          "SUBCATEGORY": null,
          "TYPE": "Discretionary",
          "geometry": {
            "coordinates": [
              -89.3864399999999,
              43.07313
            ],
            "type": "Point"
          }
        },
        {
          "ADDRESS_GEOCODE": "Pittsfield, MA",
          "AGENCY_NAME": "Environmental Protection Agency",
          "BUREAU_NAME": "Office of Air and Radiation",
          "CATEGORY": "Environmental Remediation",
          "CITY": "Pittsfield",
          "COUNTY": null,
          "FUNDING": "300131",
          "FUNDING_NUMERIC": 300131.0,
          "FUNDING_SOURCE": "IRA",
          "ID": "S-119",
          "LATITUDE": 42.44776,
          "LONGITUDE": -73.25413,
          "OBJECTID": 2,
          "PROGRAM_NAME": "Funding to Address Air Pollution: Fenceline Air Monitoring",
          "PROJECT_NAME": "Berkshire Environmental Action Team",
          "STATE_ABBR": "MA",
          "STATE_NAME": "Massachusetts",
          "SUBCATEGORY": null,
          "TYPE": "Discretionary",
          "geometry": {
            "coordinates": [
              -73.25413,
              42.4477600000001
            ],
            "type": "Point"
          }
        },
        {
          "ADDRESS_GEOCODE": "Charleston, SC",
          "AGENCY_NAME": "Environmental Protection Agency",
          "BUREAU_NAME": "Office of Air and Radiation",
          "CATEGORY": "Environmental Remediation",
          "CITY": "Charleston",
          "COUNTY": null,
          "FUNDING": "499715",
          "FUNDING_NUMERIC": 499715.0,
          "FUNDING_SOURCE": "IRA",
          "ID": "S-127",
          "LATITUDE": 32.78115,
          "LONGITUDE": -79.9316,
          "OBJECTID": 3,
          "PROGRAM_NAME": "Funding to Address Air Pollution: Fenceline Air Monitoring",
          "PROJECT_NAME": "Charleston Community Research to Action Board",
          "STATE_ABBR": "SC",
          "STATE_NAME": "South Carolina",
          "SUBCATEGORY": null,
          "TYPE": "Discretionary",
          "geometry": {
            "coordinates": [
              -79.9316,
              32.78115
            ],
            "type": "Point"
          }
        },
        {
          "ADDRESS_GEOCODE": "New Bedford, MA",
          "AGENCY_NAME": "Environmental Protection Agency",
          "BUREAU_NAME": "Office of Air and Radiation",
          "CATEGORY": "Environmental Remediation",
          "CITY": "New Bedford",
          "COUNTY": null,
          "FUNDING": "391822",
          "FUNDING_NUMERIC": 391822.0,
          "FUNDING_SOURCE": "IRA",
          "ID": "S-129",
          "LATITUDE": 41.6378,
          "LONGITUDE": -70.93089,
          "OBJECTID": 4,
          "PROGRAM_NAME": "Funding to Address Air Pollution: Fenceline Air Monitoring",
          "PROJECT_NAME": "City of New Bedford",
          "STATE_ABBR": "MA",
          "STATE_NAME": "Massachusetts",
          "SUBCATEGORY": null,
          "TYPE": "Discretionary",
          "geometry": {
            "coordinates": [
              -70.93089,
              41.6378
            ],
            "type": "Point"
          }
        },
        {
          "ADDRE

## Example 5: Interactive Features - Drawing, Search, and Popups

This section demonstrates interactive features: **drawing tools**, **geocoding/search**, and **clickable popups**. We show both **leafmap** (ipyleaflet backend) and **folium** as alternatives.

### 5A. Leafmap (ipyleaflet backend) - Richest interactivity

Leafmap's ipyleaflet backend provides the most interactive features including drawing tools, search, and clickable markers.

In [8]:
# Drawing tools with leafmap - allows drawing points, lines, polygons, rectangles, circles
m = leafmap.Map(center=[40, -100], zoom=4, draw_control=True)
m

Map(center=[40, -100], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

**Interpretation**: The drawing control (top-left) lets users draw points, lines, polygons, rectangles, and circles. Use the toolbar to select a drawing tool, then click on the map to draw. The exported GeoJSON can be saved for further analysis.

In [9]:
# Geocoding search - search for addresses/locations using Nominatim
m = leafmap.Map(draw_control=False)
url = "https://nominatim.openstreetmap.org/search?format=json&q={s}"
m.add_search_control(url)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

**Interpretation**: The search control (top-right) uses OpenStreetMap's Nominatim service to geocode addresses. Type an address or place name and press Enter. The map will zoom to the location. Note: Follow Nominatim's usage policy.

In [11]:
import leafmap
import ipywidgets as widgets

m = leafmap.Map(center=[40, -100], zoom=4)
locations = [
    [40.7128, -74.0060, "New York City", "Largest city in the US"],
    [34.0522, -118.2437, "Los Angeles", "Second largest city"],
    [41.8781, -87.6298, "Chicago", "Windy city"],
    [29.7604, -95.3698, "Houston", "Largest in Texas"],
    [33.4484, -112.0740, "Phoenix", "Fifth largest city"],
]

for lat, lon, name, desc in locations:
    # Wrap HTML string in an HTML widget
    popup_widget = widgets.HTML(value=f"<b>{name}</b><br>{desc}")
    m.add_marker(
        location=[lat, lon],
        popup=popup_widget,
        tooltip=name
    )
m

Map(center=[40, -100], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

**Interpretation**: Click on markers to see popups with HTML content. Hover over markers to see tooltips. This pattern is useful for displaying location-specific information.

**Interpretation**: Combining all interactive features in one map. Use the drawing tools to create new features, search to find locations, and click markers for information.

### 5B. Folium - Alternative with drawing and popups

Folium provides a simpler alternative using Leaflet.js. It supports drawing tools and popups but has more limited search capabilities.

In [14]:
# Drawing tools with folium
import folium
from folium.plugins import Draw

m = folium.Map(location=[40, -100], zoom_start=4)
Draw(export=True, position="topleft").add_to(m)
m

**Interpretation**: Folium's Draw plugin provides drawing tools. The `export=True` parameter adds an "Export" button to download drawn features as GeoJSON.

In [15]:
# Markers with popups in folium
m = folium.Map(location=[40, -100], zoom_start=4)
locations = [
    [40.7128, -74.0060, "New York City", "Largest city in the US"],
    [34.0522, -118.2437, "Los Angeles", "Second largest city"],
    [41.8781, -87.6298, "Chicago", "Windy city"],
]
for lat, lon, name, desc in locations:
    folium.Marker(
        location=[lat, lon],
        popup=f"<b>{name}</b><br>{desc}",
        tooltip=name
    ).add_to(m)
m

**Interpretation**: Folium markers support both `popup` (click to show) and `tooltip` (hover to show) parameters. Popups can contain HTML content.

In [16]:
# Combined: Drawing + Markers in folium
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4)
Draw(export=True).add_to(m)
folium.Marker(
    location=[40.7128, -74.0060],
    popup="<b>New York City</b><br>Largest city in the US",
    tooltip="NYC"
).add_to(m)
folium.Marker(
    location=[34.0522, -118.2437],
    popup="<b>Los Angeles</b><br>Second largest city",
    tooltip="LA"
).add_to(m)
m

**Interpretation**: Combining drawing tools with markers in folium. Note: Folium doesn't have built-in geocoding; external services or APIs (like Nominatim) would need to be called programmatically.

## Summary

| Example | Tools | Use case |
|---------|-------|----------|
| Raster | `leafmap.add_cog_layer` | Satellite imagery, remote COG streaming |
| Vector | `leafmap.add_geojson` | Points, lines, polygons with attributes |
| Combined | `add_basemap` + `add_geojson` | Raster backdrop with vector overlay |
| 3D | `leafmap.deck` + `extruded` / `ColumnLayer` | Extruded polygons, 3D columns by attribute |
| Interactive | `leafmap` + `folium` + `Draw` | Drawing tools, geocoding, popups |